# Week 3 — V1 Data Pipeline (Pathway A: PDF/TXT)

**Project:** FYPAdvisor — AI FYP Topic Recommendation Assistant
**Lab:** AI Engineering Topic 3 — Data Engineering for AI: Pipelines, Lineage and Workflows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Efxn-spec/ai-engineering-fyp-topic-assistant/blob/main/notebooks/week3_pipeline.ipynb)

**Selected pathway: A — PDF/TXT only.** The Week 2 V1 architecture has one data component, *Departmental FYP Knowledge Sources*. In V1 that component is a set of official UTM PDF documents that the assistant must retrieve from and cite (FR1, FR3, FR4/RAI1). No Topic 2 requirement needs tabular, image, JSON/API, audio or time-series data, so Pathways B–F are **intentionally not run**.

**Execution order (Lab B1):** Cell 1 → Cell 2 (Colab only) → Cell 3 → A3 → Cell 4 → Cell 4A → Cell 5A1 → Cell 5A2 → B3 → B4.1 → Cell 6A → Cell 6B → Cell 7 → Cell 8A → Cell 9 → Cell 9B

**Where it runs**
* **Google Colab:** uses the standard `/content/week3_project` workspace. Cell 2 copies the approved PDFs from this GitHub repository into `data/raw/fyp_documents/`.
* **Local checkout:** run from the repository's `notebooks/` folder. Paths resolve to the repository's own `data/` folder, so the outputs land where Lab C4 expects them.

**Adaptations to the starter code** (each is commented where it happens):
1. The corpus has three PDFs, so Part B processes every explicitly selected source, not just one file.
2. A third chunking configuration, *page-paragraph-aware*, is compared. It runs the lab's paragraph-aware splitter **per page**, so every chunk keeps a page number for FR4 citations.
3. Cell 7 can ingest a *documents* folder (the starter only expanded image folders). Manifests store paths relative to the project root, so they stay portable.

## Part A — Start with the Data

### Step A1 — From Topic 2 requirements to data needs

| Requirement ID | Needed information / feature | Candidate source | Why usable / constraint |
|---|---|---|---|
| **FR1** — answer FYP handbook rules, milestones, deliverables, assessment | Official FYP/UGP procedures: topic proposal, UGP 1/UGP 2 structure, supervisor and student roles, report and seminar requirements | S01 `utm_fyp_psm_guideline.pdf`; S02 `utm_project_scoping_handbook.pdf` | Official UTM documents. **Constraint:** S01 is the *Mechanical Engineering* UGP guideline and S02 is a *Master's* project handbook. They stand in until the Faculty of Computing PSM guideline is obtained. |
| **FR3** — critique draft title / problem statement / objectives for scope | Title rules (≤ 15 words, no redundant phrases), proposal structure (problem statement, objectives, scope) | S03 `utm_thesis_manual_2015.pdf` §1.5.1; S02 proposal-structure section | Authoritative title rules. **Constraint:** the 2015 manual targets graduate theses. |
| **FR4 / RAI1** — cite document title and section/page for every grounded answer | Source identity (title, version, SHA-256) **and page number** for every prepared passage | Same sources plus their metadata | Page-level provenance is required. This drives the Part B choice of a page-aware chunker. |
| **NFR4** — handbook update ingested in < 10 min without code changes | Repeatable, re-runnable ingestion with checksums | Part C `pipeline_v1` run | Raw PDFs are kept unchanged in `data/raw/`, so they can be re-processed. |

**Not yet available:** FR2 also needs supervisor research profiles and past FYP titles/abstracts. These are not in the repository yet and are recorded as a data gap in `docs/week3_data_inventory.md`.

### Step A2 — Set up the workspace

**REQUIRED — Cell 1.** Run first, and again after any runtime reset.

In [1]:
# Cell 1 — Workspace setup (RUN FIRST)
# Adaptation: in Colab, use the standard /content/week3_project workspace.
# In a local run from the cloned repository, use the repository root, so
# prepared outputs land directly in data/processed and data/pipeline_v1.
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    BASE_DIR = Path("/content/week3_project")
else:
    BASE_DIR = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()).resolve()

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Compatibility aliases for notebook examples
RAW = RAW_DIR
PROCESSED = PROCESSED_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR


def rel(path):
    """Path relative to BASE_DIR, so printed evidence and manifests stay portable."""
    try:
        return Path(path).resolve().relative_to(BASE_DIR.resolve()).as_posix()
    except ValueError:
        return str(path)


print("Environment:", "Google Colab" if IN_COLAB else "Local repository checkout")
print("Raw folder:", rel(RAW_DIR))
print("Processed folder:", rel(PROCESSED_DIR))

Environment: Local repository checkout
Raw folder: data/raw
Processed folder: data/processed


**OPTIONAL — Cell 2.** This replaces the dummy-ZIP extraction. The project uses its own approved PDFs, so in Colab this cell copies them from the project's GitHub repository into `data/raw/fyp_documents/`. In a local checkout the files are already there, and nothing is copied.

In [2]:
# Cell 2 — OPTIONAL: Fetch the approved project PDFs (Colab only)
import shutil
import subprocess

REPO_URL = "https://github.com/Efxn-spec/ai-engineering-fyp-topic-assistant.git"
DOC_DIR = RAW_DIR / "fyp_documents"

if IN_COLAB and not any(DOC_DIR.glob("*.pdf")):
    clone_dir = Path("/content/fyp_repo_src")
    if not clone_dir.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(clone_dir)],
            check=True,
        )
    DOC_DIR.mkdir(parents=True, exist_ok=True)
    # Works with both the Week 3 layout (data/raw/fyp_documents) and the
    # earlier layout where the PDFs sat directly in data/.
    for pdf in sorted(clone_dir.glob("data/**/*.pdf")):
        shutil.copy2(pdf, DOC_DIR / pdf.name)
        print("Copied:", pdf.name)
else:
    print("No copy needed. Using existing files in:", rel(DOC_DIR))

No copy needed. Using existing files in: data/raw/fyp_documents


**REQUIRED — Cell 3.** Confirm that the inputs exist and are accessible. A clicked upload button is not evidence; this listing is.

In [3]:
# Cell 3 — Verify uploaded inputs
files = sorted(f for f in RAW_DIR.rglob("*") if f.is_file())
print("Input count:", len(files))
for f in files:
    print(f.relative_to(RAW_DIR), "bytes:", f.stat().st_size)

assert files, "Upload at least one relevant input file before proceeding."
print("Upload verified. Select your relevant input(s) for Step A3.")

Input count: 3
fyp_documents/utm_fyp_psm_guideline.pdf bytes: 295726
fyp_documents/utm_project_scoping_handbook.pdf bytes: 602977
fyp_documents/utm_thesis_manual_2015.pdf bytes: 1616473
Upload verified. Select your relevant input(s) for Step A3.


### Step A3 — Identify data type and inspection evidence

**Data type:** unstructured text (PDF). **Processing representation:** page-level text, then text chunks with source metadata.

All three sources are inspected. For each one, the cell shows extractability, layout noise, and a passage that a Topic 2 requirement depends on. A file merely existing is not evidence of ingestion.

In [4]:
# Step A3 — Inspect the selected data sources (REQUIRED)
import importlib.util
import subprocess
if importlib.util.find_spec("pypdf") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pypdf"], check=True)

import re
from collections import Counter

import pandas as pd
from pypdf import PdfReader

# Source identity, also recorded in docs/week3_data_inventory.md (Step A4)
SOURCE_INFO = {
    "fyp_documents/utm_fyp_psm_guideline.pdf": {
        "source_id": "S01",
        "title": "Undergraduate Project (UGP) Guideline, Faculty of Mechanical Engineering, UTM",
        "version": "UGP Guideline/2014/1 (2014)",
        "owner": "Undergraduate Project Committee, Faculty of Mechanical Engineering, UTM",
        "access": "Publicly downloadable UTM document; (c) UTM; redistribution permission not verified",
        "requirements": "FR1, FR3, FR4/RAI1",
        "probe": "TYPE, SCOPE AND STRUCTURE OF PROJECT",
    },
    "fyp_documents/utm_project_scoping_handbook.pdf": {
        "source_id": "S02",
        "title": "Student Handbook for Master's Project, MSc Business Intelligence and Analytics, UTM",
        "version": "1st Edition 2017/2018",
        "owner": "MSc BIA Programme Coordinator, Advanced Informatics School (AIS), UTM",
        "access": "Publicly downloadable UTM document; (c) UTM; redistribution permission not verified",
        "requirements": "FR1, FR3, FR4/RAI1",
        "probe": "Research scope",
    },
    "fyp_documents/utm_thesis_manual_2015.pdf": {
        "source_id": "S03",
        "title": "UTM Thesis Manual, School of Graduate Studies",
        "version": "2015 edition",
        "owner": "School of Graduate Studies (SPS), UTM",
        "access": "Public URL (research.utm.my); (c) UTM; redistribution permission not verified",
        "requirements": "FR3, FR4/RAI1",
        "probe": "15 words",
    },
}

LETTER_SPACED = re.compile(r"\b(?:[A-Za-z] ){4,}[A-Za-z]\b")

inspection_rows = []
probe_evidence = []
for rel_path, info in SOURCE_INFO.items():
    path = RAW_DIR / rel_path
    assert path.is_file(), f"Missing source: {path}"
    reader = PdfReader(str(path))
    page_texts = [page.extract_text() or "" for page in reader.pages]
    page_chars = [len(t.strip()) for t in page_texts]

    # Running header/footer candidates: the same short line (digits ignored)
    # appearing on at least 30% of pages.
    line_pages = Counter()
    for t in page_texts:
        lines = {re.sub(r"\d+", "#", ln.strip()) for ln in t.splitlines()}
        line_pages.update(ln for ln in lines if 8 <= len(ln) <= 60)
    repeated = [ln for ln, n in line_pages.most_common(3) if n >= 0.3 * len(page_texts)]

    inspection_rows.append({
        "id": info["source_id"],
        "file": path.name,
        "bytes": path.stat().st_size,
        "pages": len(page_texts),
        "chars": sum(page_chars),
        "min_chars/page": min(page_chars),
        "near_empty_pages": sum(c < 20 for c in page_chars),
        "pdf_meta_title": (reader.metadata or {}).get("/Title"),
        "letter_spaced_fragments": sum(len(LETTER_SPACED.findall(t)) for t in page_texts),
        "running_header_candidates": repeated,
    })

    for page_no, t in enumerate(page_texts, 1):
        pos = t.find(info["probe"])
        if pos >= 0:
            probe_evidence.append((info["source_id"], path.name, page_no,
                                   re.sub(r"\s+", " ", t[max(0, pos - 80):pos + 260])))
            break

inspection_df = pd.DataFrame(inspection_rows)
pd.set_option("display.max_colwidth", 70)
print("--- Source inspection (all selected sources) ---")
print(inspection_df.to_string(index=False))

print("\n--- Requirement-relevant passages (first match per source) ---")
for sid, name, page_no, snippet in probe_evidence:
    print(f"\n[{sid}] {name} | page {page_no}\n{snippet!r}")

assert len(probe_evidence) == len(SOURCE_INFO), "A requirement-relevant passage was not found."
assert (inspection_df["near_empty_pages"] == 0).all(), "Some pages may need OCR."
print("\nInspection complete: all sources have extractable text; no OCR required.")

--- Source inspection (all selected sources) ---
 id                             file   bytes  pages  chars  min_chars/page  near_empty_pages                        pdf_meta_title  letter_spaced_fragments                 running_header_candidates
S01        utm_fyp_psm_guideline.pdf  295726     23  35857              60                 0                            SeliaanPSM                        0                       [UGP Guideline/#/#]
S02 utm_project_scoping_handbook.pdf  602977     26  26582             263                 0 STUDENT HANDBOOK FOR MASTER’S PROJECT                       58 [Student Handbook for Master’s Project #]
S03       utm_thesis_manual_2015.pdf 1616473     80  92870             114                 0       Microsoft Word - B I_edited.doc                        0                                        []

--- Requirement-relevant passages (first match per source) ---

[S01] utm_fyp_psm_guideline.pdf | page 3
' complete the project individually and ethically in 

**A3 observations** (also in `docs/week3_data_inventory.md`):
* All 129 pages across the three PDFs have a text layer; none are scanned images, so **OCR is not required**.
* The PDF metadata titles are not reliable (for example, `Microsoft Word - B I_edited.doc`), so document titles are curated in `SOURCE_INFO`.
* Layout noise is present: a running header/footer on most pages (`UGP Guideline/#/#`, `Student Handbook for Master's Project #`) and letter-spaced headings in S02 (`I n t r o d u c t i o n`). These are recorded for Topic 4 rather than aggressively cleaned now.
* **Relevance caveat:** S01 is the Faculty of *Mechanical Engineering* UGP guideline, and S02 is a *postgraduate* project handbook. Both are interim proxies for the Faculty of Computing undergraduate PSM documents.

### Step A4 — Data inventory → `docs/week3_data_inventory.md`

**Checkpoint A:** FR1/FR3 need the rule text, and FR4/RAI1 need title + page provenance. The loaded sample above shows real passages, such as the 15-word title rule on page 6 of S03, not just a "file loaded" message.

## Part B — Prepare the Data

### Step B1 — Select the preparation pathway

| Pathway | Run? | Reason |
|---|---|---|
| **A PDF/TXT** | **Yes** | All V1 knowledge sources are PDFs (FR1, FR3, FR4). |
| B CSV/Excel | No | No tabular source in V1. A past-FYP-titles list could become tabular later (FR2). |
| C Images | No | No image input or vision capability. |
| D JSON/API | No | No approved API source. |
| E Audio | No | Not applicable. |
| F Time-series | No | Not applicable. |

### Step B2 — Guided code

**REQUIRED — Cell 4: common metadata helpers** (source selection, SHA-256, manifest writer).

In [5]:
# Cell 4 — REQUIRED: Source Selection + Lineage Utilities
# Run ONCE after Cell 1 and Cell 3
from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone

# Select the exact source before parsing.
# Never automatically take the first file by extension.


def select_source(relative_path=None, filename=None, suffixes=None):
    if relative_path:
        selected = RAW_DIR / relative_path
        if not selected.is_file():
            raise FileNotFoundError(
                f"Source not found: {selected}"
            )
        return selected
    matches = sorted(
        p for p in RAW_DIR.rglob("*")
        if p.is_file()
        and (filename is None or p.name == filename)
        and (suffixes is None or p.suffix.lower() in suffixes)
    )
    print(
        "Candidates:",
        [str(p.relative_to(RAW_DIR)) for p in matches]
    )
    if len(matches) != 1:
        raise ValueError(
            "Select exactly one source. "
            "Set relative_path to an exact file under RAW_DIR."
        )
    return matches[0]


def source_hash(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()


def save_manifest(records):
    manifest = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "records": records
    }
    target = BASE_DIR / "data" / "week3_manifest.json"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )
    print("Saved lineage:", rel(target))
    return manifest


# Confirmation messages
print("Cell 4 completed successfully.")
print("Source selection and lineage utilities are ready.")
print("Available functions:")
print("- select_source()")
print("- source_hash()")
print("- save_manifest()")

Cell 4 completed successfully.
Source selection and lineage utilities are ready.
Available functions:
- select_source()
- source_hash()
- save_manifest()


**REQUIRED — Cell 4A: initialise the common lineage registry** (run before the Cell 5 pathway).

In [6]:
# CELL 4A — COMMON LINEAGE REGISTRY
# REQUIRED: Run once before selected Part B pathways
from pathlib import Path

lineage_entries = {}


def register_lineage(source, output, processing):
    source = Path(source).resolve()
    output = Path(output).resolve()
    # Validate source
    if not source.is_file():
        raise FileNotFoundError(
            f"Source file not found: {source}"
        )
    # Validate prepared output
    if not output.is_file():
        raise FileNotFoundError(
            f"Prepared output not found: {output}"
        )
    if output.stat().st_size == 0:
        raise ValueError(
            f"Prepared output is empty: {output}"
        )
    # Validate processing description
    if not isinstance(processing, str) or not processing.strip():
        raise ValueError(
            "A processing description is required."
        )
    processing = processing.strip()
    if processing.upper().startswith("REPLACE"):
        raise ValueError(
            "Replace the placeholder with the actual method."
        )
    # Ensure source and output belong to project workspace
    source.relative_to(RAW_DIR.resolve())
    output.relative_to(PROCESSED_DIR.resolve())
    # Same source/output pair is updated, not duplicated
    key = (str(source), str(output))
    lineage_entries[key] = {
        "source": source,
        "output": output,
        "processing": processing
    }
    print(
        "Registered:",
        source.name,
        "->",
        output.name
    )


print("Common lineage registry ready.")
print("Registered entries:", len(lineage_entries))

Common lineage registry ready.
Registered entries: 0


#### Pathway A — PDF / text: parsing, cleaning and chunking (CONDITIONAL REQUIRED: this project)

Cell 5A1 parses each explicitly selected PDF and applies the lab's cleaning rule, which only collapses spaces and tabs. It then compares three configurations at the same 500-character size:
* **fixed**: the lab's fixed-size character chunks over the whole document;
* **paragraph-aware**: the lab's paragraph-aware splitter over the whole document;
* **page-paragraph-aware** (adaptation): the same paragraph-aware splitter applied **within each page**, so every chunk keeps the page number FR4 needs.

None of these is semantic chunking. This step produces no embeddings or vector index.

In [7]:
# Cell 5A1
# PATHWAY A1 — PDF/TXT
# CONDITIONAL REQUIRED: Run only for document-based projects (this project)
import importlib.util
import subprocess
import sys
if importlib.util.find_spec("pypdf") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pypdf"], check=True)

import re

import pandas as pd
from pypdf import PdfReader

# Step 1: Select the exact source files (explicit paths, never "first PDF found")
SELECTED_SOURCES = [
    select_source(relative_path="fyp_documents/utm_fyp_psm_guideline.pdf"),
    select_source(relative_path="fyp_documents/utm_project_scoping_handbook.pdf"),
    select_source(relative_path="fyp_documents/utm_thesis_manual_2015.pdf"),
]


# Step 2: Extract text according to file type (kept page by page)
def extract_pages(f):
    if f.suffix.lower() == ".pdf":
        reader = PdfReader(str(f))
        return [
            (i + 1, page.extract_text() or "")
            for i, page in enumerate(reader.pages)
        ]
    elif f.suffix.lower() == ".txt":
        return [(1, f.read_text(encoding="utf-8"))]
    raise ValueError("Pathway A needs a PDF or TXT source")


# Step 3: Basic text cleaning (lab rule: collapse spaces/tabs only)
def clean_text(s):
    return re.sub(r"[ \t]+", " ", s).strip()


# Step 4: Fixed-size chunking (lab version, unchanged)
def fixed_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    return [
        s[i:i + size]
        for i in range(0, len(s), size)
    ]


# Step 5: Paragraph-aware chunking (lab version, unchanged)
def paragraph_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    parts = []
    current = ""
    for segment in re.split(r"\n\s*\n", s):
        segment = segment.strip()
        if not segment:
            continue
        if len(segment) > size:
            if current:
                parts.append(current)
                current = ""
            parts.extend(fixed_chunks(segment, size))
        elif len(current) + len(segment) + (2 if current else 0) <= size:
            current += ("\n\n" if current else "") + segment
        else:
            if current:
                parts.append(current)
            current = segment
    if current:
        parts.append(current)
    return parts


# Step 5b (adaptation for FR4/RAI1): paragraph-aware chunking within each
# page, so that every chunk can be cited with its page number.
def page_paragraph_chunks(pages, size=500):
    out = []
    for page_no, content in pages:
        page_clean = clean_text(content)
        if page_clean:
            out.extend((page_no, chunk) for chunk in paragraph_chunks(page_clean, size))
    return out


CHUNK_SIZE = 500
METHODS = ["fixed", "paragraph-aware", "page-paragraph-aware"]

docs = {}
for f in SELECTED_SOURCES:
    pages = extract_pages(f)
    text = "\n".join(content for _, content in pages)
    clean = clean_text(text)
    print("Source:", f.name)
    print("  Pages/segments:", len(pages))
    print("  Extracted characters:", len(text))
    print("  Cleaned characters:", len(clean))
    print("  Content sample:", repr(clean[:200]))
    assert clean, f"No usable text in {f.name}: check extraction or OCR needs."
    docs[f] = {
        "pages": pages,
        "clean": clean,
        "fixed": [(None, c) for c in fixed_chunks(clean, CHUNK_SIZE)],
        "paragraph-aware": [(None, c) for c in paragraph_chunks(clean, CHUNK_SIZE)],
        "page-paragraph-aware": page_paragraph_chunks(pages, CHUNK_SIZE),
    }

# Step 6: Compare the preparation approaches with observable evidence
SENTENCE_END = tuple(".!?:;")
comparison = []
for method in METHODS:
    chunks = [c for d in docs.values() for c in d[method]]
    texts = [t for _, t in chunks]
    lengths = [len(t) for t in texts]
    comparison.append({
        "method": method,
        "chunks": len(texts),
        "mean_chars": round(sum(lengths) / len(lengths)),
        "min_chars": min(lengths),
        "max_chars": max(lengths),
        "chunks_<100_chars": sum(n < 100 for n in lengths),
        "start_mid_sentence_%": round(100 * sum(t.strip()[:1].islower() for t in texts) / len(texts), 1),
        "end_at_sentence_%": round(100 * sum(t.strip().endswith(SENTENCE_END) for t in texts) / len(texts), 1),
        "with_page_ref_%": round(100 * sum(p is not None for p, _ in chunks) / len(chunks), 1),
    })

comparison_df = pd.DataFrame(comparison)
print("\n--- Chunking Comparison (3 sources, size=500 chars) ---")
print(comparison_df.to_string(index=False))

# Per-source chunk counts
per_source = pd.DataFrame(
    [{"source": f.name, **{m: len(d[m]) for m in METHODS}} for f, d in docs.items()]
)
print("\n--- Chunks per source ---")
print(per_source.to_string(index=False))

# Representative samples from the thesis manual (S03)
s03 = SELECTED_SOURCES[2]
print("\nFixed sample (S03, chunk 1 -> 2 boundary):")
print("  end of chunk 1:  ", repr(docs[s03]["fixed"][0][1][-80:]))
print("  start of chunk 2:", repr(docs[s03]["fixed"][1][1][:80]))
print("\nParagraph-aware sample (S03, chunk 1):")
print(repr(docs[s03]["paragraph-aware"][0][1][:180]))
print("\nPage-paragraph-aware sample (S03, chunk containing the title rule):")
title_rule = next((p, t) for p, t in docs[s03]["page-paragraph-aware"] if "15 words" in t)
print("  page", title_rule[0], "|", repr(title_rule[1][:300]))

print("\nPATHWAY A1 completed successfully.")

Source: utm_fyp_psm_guideline.pdf
  Pages/segments: 23
  Extracted characters: 36087
  Cleaned characters: 35964
  Content sample: 'UNDERGARDUATE PROJECT (UGP) \nGUIDELINE \n \nFACULTY OF MECHANICAL ENGINEERING, \nUNIVERSITI TEKNOLOGI MALAYSIA \n \n \n \n \n \n \n \n \n \nPrepared by: \nUndergraduate Project Committee \n \n \n \n \n2014 \n\n \nUGP Guide'


Source: utm_project_scoping_handbook.pdf
  Pages/segments: 26
  Extracted characters: 26725
  Cleaned characters: 26483
  Content sample: 'MSc Business \nIntelligence and \nAnalytics \nDr. Nurulhuda FIrdaus Bt Mohd Azmi \n(Programme Coordinator) \nDr. Suraya Bt Ya’acob, Dr. Doris Wong, Dr. \nNilam Nur Bt Amir Sjarif, Dr. Saiful Adli B. \nIsmail'


Source: utm_thesis_manual_2015.pdf
  Pages/segments: 80
  Extracted characters: 93647
  Cleaned characters: 87300
  Content sample: 'innovative ● entrepreneurial ● global \n \nUTM Thesis Manual \nSchool of Graduate Studies \nSekolah Pengajian Siswazah \n \n2 \n \n \n \n \n \nTHESIS MANUAL \n \n \n \n \n \n \n \nCHAPTER 1 \n \n \n \n \n \n \nORGANISATION OF T'

--- Chunking Comparison (3 sources, size=500 chars) ---
              method  chunks  mean_chars  min_chars  max_chars  chunks_<100_chars  start_mid_sentence_%  end_at_sentence_%  with_page_ref_%
               fixed     300         499        300        500                  0                  71.7                6.7              0.0
     paragraph-aware     381         375          1        500                 28                  26.2               30.7              0.0
page-paragraph-aware     420         340          1        500                 48                  22.9               36.0            100.0

--- Chunks per sourc

**REQUIRED IF PATHWAY A — Cell 5A2: export document records.** The configuration is chosen from the A1 evidence (see the B3 comparison below and `docs/week3_pipeline_decisions.md`).

In [8]:
# Cell 5A2
# PATHWAY A2 — REQUIRED after A1
# Choose and export traceable text chunks
import json

# Step 1: Select the preparation approach (justified in B3)
CHOSEN_METHOD = "page-paragraph-aware"
if CHOSEN_METHOD in METHODS:
    approach = f"{CHOSEN_METHOD}-char-{CHUNK_SIZE}"
else:
    raise ValueError(f"Choose one of {METHODS}")

# Step 2: Create traceable records (one per chunk, with source identity + page)
records = []
for f, d in docs.items():
    info = SOURCE_INFO[f.relative_to(RAW_DIR).as_posix()]
    sha = source_hash(f)
    for i, (page, chunk) in enumerate(d[CHOSEN_METHOD], 1):
        records.append({
            "record_id": f"{f.stem}-{i:04d}",
            "text": chunk,
            "source": f.relative_to(RAW_DIR).as_posix(),
            "source_id": info["source_id"],
            "document_title": info["title"],
            "page": page,  # physical PDF page (1-based); None for whole-document splitters
            "source_sha256": sha,
            "processing": approach,
        })

# Step 3: Export prepared records
prepared_path = PROCESSED_DIR / "prepared_text.jsonl"
prepared_path.parent.mkdir(parents=True, exist_ok=True)
with prepared_path.open("w", encoding="utf-8") as output:
    for row in records:
        output.write(json.dumps(row, ensure_ascii=False) + "\n")

# Step 4: Basic validation
assert prepared_path.is_file(), "Prepared output not found"
assert prepared_path.stat().st_size > 0, "Prepared output is empty"
assert records, "No text chunks were produced"
assert all(row["text"].strip() for row in records), "One or more chunks are empty"
assert len({row["record_id"] for row in records}) == len(records), "Duplicate record IDs"
if CHOSEN_METHOD == "page-paragraph-aware":
    assert all(isinstance(row["page"], int) for row in records), "A chunk lost its page reference"
with prepared_path.open("r", encoding="utf-8") as file:
    exported_records = [json.loads(line) for line in file if line.strip()]
assert len(exported_records) == len(records), "Exported record count does not match"
assert exported_records == records, "Exported content differs from prepared records"

# Step 5: Display evidence
print("PATHWAY A2 completed successfully.")
print("Selected approach:", approach)
print("Total chunks exported:", len(records))
print("Prepared file:", rel(prepared_path))
print("File size:", prepared_path.stat().st_size, "bytes")
print("Validation: PASS")
print("First record:", json.dumps(records[0], ensure_ascii=False)[:400])

PATHWAY A2 completed successfully.
Selected approach: page-paragraph-aware-char-500
Total chunks exported: 420
Prepared file: data/processed/prepared_text.jsonl
File size: 298319 bytes
Validation: PASS
First record: {"record_id": "utm_fyp_psm_guideline-0001", "text": "UNDERGARDUATE PROJECT (UGP) \nGUIDELINE\n\nFACULTY OF MECHANICAL ENGINEERING, \nUNIVERSITI TEKNOLOGI MALAYSIA\n\nPrepared by: \nUndergraduate Project Committee\n\n2014", "source": "fyp_documents/utm_fyp_psm_guideline.pdf", "source_id": "S01", "document_title": "Undergraduate Project (UGP) Guideline, Faculty of Mechanical Engineering, UTM", "page


### Step B3 — Validate the prepared result and compare one decision

**REQUIRED — basic output contract.** This is a stage contract, not a full quality assessment; Topic 4 covers that. The representative records are also opened below, because a non-empty file alone does not prove the content is correct.

In [9]:
# B3 BASIC OUTPUT CONTRACT — REQUIRED FOR ALL PATHWAYS
assert prepared_path.exists(), 'Prepared output missing.'
assert prepared_path.stat().st_size > 0, 'Prepared output is empty.'
print('Output:', rel(prepared_path), 'Size:', prepared_path.stat().st_size)
print('Validation: file exists and is non-empty; verify content manually too.')

# Manual content check against FR4/RAI1: can a requirement-relevant passage
# be cited with document title + page from the prepared records?
print("\n--- Representative prepared records ---")
for probe in ["15 words", "TYPE, SCOPE AND STRUCTURE", "Research scope"]:
    hit = next(r for r in exported_records if probe in r["text"])
    print(f"\n[{hit['source_id']}] {hit['document_title']} | page {hit['page']} | {hit['record_id']}")
    print(repr(hit["text"][:260]))

print("\n--- Records per source ---")
print(pd.Series([r["source_id"] for r in exported_records]).value_counts().sort_index().to_string())

Output: data/processed/prepared_text.jsonl Size: 298319
Validation: file exists and is non-empty; verify content manually too.

--- Representative prepared records ---

[S03] UTM Thesis Manual, School of Graduate Studies | page 6 | utm_thesis_manual_2015-0016
'6 \n1.5.1 Thesis Title\n\ni. Thesis title should be a concise description of the main focus and \ncontribution of the research. It should not contain more than 15 words \nexcluding grammatical words such as articles, conjunction and prepositions.\n\nii. To avoid redu'

[S01] Undergraduate Project (UGP) Guideline, Faculty of Mechanical Engineering, UTM | page 3 | utm_fyp_psm_guideline-0013
'6.0 TYPE, SCOPE AND STRUCTURE OF PROJECT \n6.1 Type of Project \nA UGP topic should have the criteria through which the objectives and aims \nof UGP can be achieved. It should be related to engineering field and could be'

[S02] Student Handbook for Master's Project, MSc Business Intelligence and Analytics, UTM | page 14 | utm_project_scoping_h

**B3 comparison** (actual values are in the Cell 5A1 output; summarised in `docs/week3_pipeline_decisions.md`):

| Approach / Configuration A | Approach / Configuration B | Observed evidence | Selected approach and trade-off |
|---|---|---|---|
| Fixed 500-char chunks over the whole document (lab baseline) | Paragraph-aware 500-char chunks, **applied per page** (page-paragraph-aware) | Fixed: chunks often start mid-sentence or mid-word, and no chunk has a page reference. Page-paragraph-aware: every chunk carries its page number, and boundaries follow paragraphs, so far fewer chunks start mid-sentence. The cost is more, shorter chunks. | **Page-paragraph-aware.** Benefit: FR4/RAI1 page-level citations become possible, and passages such as the 15-word title rule (S03 p.6) stay intact. Accepted limitation: a paragraph that runs across a page break is split in two, and running headers stay inside chunks. |

The whole-document paragraph-aware splitter was also measured. Its boundaries are similar, but it loses page provenance, so it cannot meet FR4.

### Step B4 — Preserve metadata, lineage and export

**B4.1 — Register Pathway A** (one entry per source → the shared prepared corpus).

In [10]:
# Register Pathway A — PDF/TXT (one lineage entry per selected source)
for f in SELECTED_SOURCES:
    register_lineage(
        source=f,
        output=prepared_path,
        processing=approach
    )
print("Registered entries:", len(lineage_entries))

Registered: utm_fyp_psm_guideline.pdf -> prepared_text.jsonl
Registered: utm_project_scoping_handbook.pdf -> prepared_text.jsonl
Registered: utm_thesis_manual_2015.pdf -> prepared_text.jsonl
Registered entries: 3


**B4.2 — Cell 6A: generate the common lineage manifest.** The source version and access status are filled from the inspected source identity (`SOURCE_INFO`), not left as placeholders.

In [11]:
# CELL 6A — GENERATE COMMON LINEAGE MANIFEST
# REQUIRED: Run after all selected pathways are registered
assert lineage_entries, (
    "No lineage entries registered. "
    "Register at least one selected pathway first."
)
lineage_records = []

for entry in lineage_entries.values():
    source = entry["source"]
    output = entry["output"]
    processing = entry["processing"]

    # Validate registered files again
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert output.stat().st_size > 0, "Empty output"

    source_rel = source.relative_to(RAW_DIR.resolve()).as_posix()
    info = SOURCE_INFO[source_rel]
    lineage_records.append({
        "source_id": info["source_id"],
        "source_filename": source_rel,
        "source_title": info["title"],
        "source_sha256": source_hash(source),
        "prepared_output": output.relative_to(BASE_DIR.resolve()).as_posix(),
        "prepared_sha256": source_hash(output),
        "prepared_records_from_source": sum(r["source"] == source_rel for r in records),
        "processing_note": processing,
        "source_version": info["version"],
        "source_owner": info["owner"],
        "access_rights": info["access"],
        "known_limitations": (
            "page = physical PDF page index; paragraphs crossing a page break are split; "
            "running headers/footers and letter-spaced headings retained"
        ),
        "status": "Prepared; quality review pending"
    })

# Reuse the common manifest writer from Cell 4
manifest = save_manifest(lineage_records)

print("\nCOMMON LINEAGE MANIFEST GENERATED")
print("Total lineage entries:", len(manifest["records"]))
print("Saved to:", rel(BASE_DIR / "data" / "week3_manifest.json"))

Saved lineage: data/week3_manifest.json

COMMON LINEAGE MANIFEST GENERATED
Total lineage entries: 3
Saved to: data/week3_manifest.json


**B4.3 — Cell 6B: verify the saved manifest.**

In [12]:
# CELL 6B — VERIFY COMMON LINEAGE MANIFEST
# REQUIRED: Run after Cell 6A
import json

manifest_path = BASE_DIR / "data" / "week3_manifest.json"
assert manifest_path.is_file(), (
    "Common lineage manifest not found."
)
with manifest_path.open("r", encoding="utf-8") as file:
    saved_manifest = json.load(file)

saved_records = saved_manifest.get("records", [])
assert saved_records, (
    "No lineage records found in saved manifest."
)
assert len(saved_records) == len(lineage_entries), (
    "Saved lineage count does not match registration."
)
for record in saved_records:
    source = RAW_DIR / record["source_filename"]
    output = BASE_DIR / record["prepared_output"]
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert source_hash(source) == record["source_sha256"], (
        f"Source checksum mismatch: {source}"
    )
    assert source_hash(output) == record["prepared_sha256"], (
        f"Prepared output checksum mismatch: {output}"
    )

print("COMMON LINEAGE VALIDATION: PASS")
print("Verified records:", len(saved_records))
print("Manifest:", rel(manifest_path))

COMMON LINEAGE VALIDATION: PASS
Verified records: 3
Manifest: data/week3_manifest.json


## Part C — Build and Document the V1 Data Pipeline

| Code | Status | Run here? |
|---|---|---|
| Cell 7 | REQUIRED | Yes |
| **Cell 8A** | CONDITIONAL REQUIRED (PDF/TXT) | **Yes** |
| Cells 8B–8F | CONDITIONAL REQUIRED (other data types) | No (not this project's data types) |
| Cell 9 | REQUIRED | Yes |

### Step C1 — Implement and run the integrated V1 data pipeline

**Cell 7 — REQUIRED: common pipeline utilities.** Stages: ingest → inspect → prepare/store → validate → record lineage.

In [13]:
# Cell 7 — REQUIRED: Common V1 Pipeline Utilities
# Run ONCE after completing Part B
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import sys

# Step 1: Define standard project paths (same resolution rule as Cell 1)
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    BASE_DIR = Path("/content/week3_project")
else:
    BASE_DIR = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()).resolve()
RAW_DIR = BASE_DIR / "data" / "raw"
PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"

PIPE_DIR.mkdir(parents=True, exist_ok=True)

# Store registered pipeline runs for repeatability verification
PIPELINE_RUNS = {}

# Adaptation: the starter only expanded folders for images. This project's
# corpus is a folder of PDFs, so folder ingestion is defined per data type.
FOLDER_SUFFIXES = {
    "images": {".png", ".jpg", ".jpeg", ".webp"},
    "documents": {".pdf", ".txt"},
}

SOURCE_VERSION_NOTE = "Per source in data/week3_manifest.json and docs/week3_data_inventory.md"
ACCESS_NOTE = ("Publicly downloadable UTM documents; redistribution permission "
               "not verified — check before sharing")


def rel(path):
    try:
        return Path(path).resolve().relative_to(BASE_DIR.resolve()).as_posix()
    except ValueError:
        return str(path)


# Step 2: Calculate SHA-256 checksum
def sha256_file(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()


# Step 3: Resolve an explicitly selected source
def resolve_source(relative_path, folder=False):
    source = RAW_DIR / relative_path
    if folder:
        if not source.is_dir():
            raise FileNotFoundError(
                f"Source folder not found: {source}"
            )
    else:
        if not source.is_file():
            raise FileNotFoundError(
                f"Source file not found: {source}"
            )
    return source


# Step 4: Execute the selected V1 data pipeline
def execute_pipeline(kind, source, inspect_fn, prepare_fn, configuration):
    source = Path(source)

    # INGEST: Identify relevant source files
    if source.is_dir():
        suffixes = FOLDER_SUFFIXES.get(kind)
        if suffixes is None:
            raise ValueError(f"No folder ingestion rule for data type: {kind}")
        inputs = sorted(
            p for p in source.iterdir()
            if p.is_file() and p.suffix.lower() in suffixes
        )
    else:
        inputs = [source]

    if not inputs or any(not p.is_file() for p in inputs):
        raise ValueError(
            "No valid source files to ingest."
        )

    # INSPECT: Apply data-type-specific inspection
    inspection = inspect_fn(inputs, configuration)

    # Define output directory
    output_dir = PIPE_DIR / kind
    output_dir.mkdir(parents=True, exist_ok=True)

    # PREPARE / STORE: Apply selected transformation
    outputs, mappings, method = prepare_fn(
        inputs,
        output_dir,
        configuration
    )
    outputs = [Path(p) for p in outputs]

    # VALIDATE: Check prepared outputs
    if not outputs:
        raise ValueError(
            "No prepared outputs were produced."
        )
    for output in outputs:
        if not output.is_file():
            raise FileNotFoundError(
                f"Prepared output missing: {output}"
            )
        if output.stat().st_size == 0:
            raise ValueError(
                f"Prepared output is empty: {output}"
            )

    # Record basic validation evidence
    validation = {
        "output_count": len(outputs),
        "output_sha256": [sha256_file(p) for p in outputs],
        "output_bytes": [p.stat().st_size for p in outputs]
    }

    # RECORD LINEAGE: Create pipeline execution record
    record = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "data_type": kind,
        "configuration": configuration,
        "processing_method": method,
        "sources": [
            {"path": rel(p), "sha256": sha256_file(p)}
            for p in inputs
        ],
        "outputs": [rel(p) for p in outputs],
        "source_output_mapping": mappings,
        "inspection": inspection,
        "validation": validation,
        "source_version": SOURCE_VERSION_NOTE,
        "access_rights": ACCESS_NOTE,
    }

    # Save pipeline manifest
    manifest_path = output_dir / "pipeline_manifest.json"
    manifest_path.write_text(
        json.dumps(record, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

    print(
        f"SUCCESS: {kind} | "
        f"Inputs: {len(inputs)} | "
        f"Outputs: {len(outputs)}"
    )
    print("Manifest:", rel(manifest_path))
    return record


# Step 5: Register and run a selected pathway
def register_and_run(kind, source, inspect_fn, prepare_fn, configuration):
    # Keep the runner for repeatability verification
    def runner():
        return execute_pipeline(
            kind,
            source,
            inspect_fn,
            prepare_fn,
            configuration
        )

    first = runner()
    PIPELINE_RUNS[kind] = {
        "first": first,
        "runner": runner
    }
    return first


print("Cell 7 ready.")
print("Run ONLY the selected Cell 8 pathway(s).")
print("Common pipeline utilities are available.")

Cell 7 ready.
Run ONLY the selected Cell 8 pathway(s).
Common pipeline utilities are available.


**Cell 8A — CONDITIONAL REQUIRED: PDF/TXT.** The source folder and configuration match Part B (`page-paragraph-aware`, 500 characters). Output: `data/pipeline_v1/documents/prepared_text.jsonl` and `pipeline_manifest.json`.

In [14]:
# Cell 8A — CONDITIONAL REQUIRED: PDF/TXT Projects
# Requires Cell 7 (Common V1 Pipeline Utilities) and SOURCE_INFO from Step A3.
import importlib.util
import subprocess
import sys
if importlib.util.find_spec("pypdf") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pypdf"], check=True)

import re
import json
from pypdf import PdfReader

# Step 1: Select the source and preparation configuration (Part B decisions)
SOURCE_A = resolve_source("fyp_documents", folder=True)

CONFIG_A = {
    "chunk_size": 500,
    "method": "page-paragraph-aware",
    "cleaning": "collapse spaces/tabs only",
}

assert "SOURCE_INFO" in globals(), "Run Step A3 first (source identity)."


def _extract_pages(src):
    if src.suffix.lower() == ".pdf":
        reader = PdfReader(str(src))
        return [(i + 1, page.extract_text() or "") for i, page in enumerate(reader.pages)]
    elif src.suffix.lower() == ".txt":
        return [(1, src.read_text(encoding="utf-8"))]
    raise ValueError("Expected a PDF or TXT source")


# Step 2: Inspect document content
def inspect_documents(inputs, cfg):
    summary = []
    for src in inputs:
        pages = _extract_pages(src)
        text = "\n".join(t for _, t in pages)
        if not text.strip():
            raise ValueError(
                f"No extractable text found in {src.name}. "
                "Scanned PDFs may require OCR."
            )
        summary.append({
            "source": src.name,
            "characters": len(text),
            "pages": len(pages) if src.suffix.lower() == ".pdf" else None,
        })
    return {"documents": summary}


def _chunk(text, size, method):
    """Same splitters as Part B: fixed or paragraph-aware."""
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    if method == "fixed":
        return [text[i:i + size] for i in range(0, len(text), size)]
    chunks = []
    current = ""
    for section in re.split(r"\n\s*\n", text):
        section = section.strip()
        if not section:
            continue
        if len(section) > size:
            if current:
                chunks.append(current)
                current = ""
            chunks.extend(section[i:i + size] for i in range(0, len(section), size))
        elif len(current) + len(section) + (2 if current else 0) <= size:
            current += ("\n\n" if current else "") + section
        else:
            if current:
                chunks.append(current)
            current = section
    if current:
        chunks.append(current)
    return chunks


# Step 3: Define document preparation
def prepare_documents(inputs, out, cfg):
    size = int(cfg["chunk_size"])
    method = cfg["method"]
    if method not in {"fixed", "paragraph-aware", "page-paragraph-aware"}:
        raise ValueError("Choose 'fixed', 'paragraph-aware' or 'page-paragraph-aware'")
    label = f"{method}-char-{size}"

    target = out / "prepared_text.jsonl"
    mapping = []
    total = 0
    with target.open("w", encoding="utf-8") as handle:
        for src in inputs:
            source_rel = src.relative_to(RAW_DIR).as_posix()
            info = SOURCE_INFO[source_rel]
            pages = _extract_pages(src)
            clean = lambda s: re.sub(r"[ \t]+", " ", s).strip()
            if method == "page-paragraph-aware":
                chunks = [
                    (page_no, c)
                    for page_no, content in pages
                    if clean(content)
                    for c in _chunk(clean(content), size, "paragraph-aware")
                ]
            else:
                whole = clean("\n".join(t for _, t in pages))
                chunks = [(None, c) for c in _chunk(whole, size, method)]
            if not chunks:
                raise ValueError(f"No text chunks produced for {src.name}")

            # Step 4: Store prepared document records
            sha = sha256_file(src)
            for i, (page_no, chunk) in enumerate(chunks, 1):
                handle.write(json.dumps({
                    "record_id": f"{src.stem}-{i:04d}",
                    "text": chunk,
                    "source": source_rel,
                    "source_id": info["source_id"],
                    "document_title": info["title"],
                    "page": page_no,
                    "source_sha256": sha,
                    "processing": label,
                }, ensure_ascii=False) + "\n")
            total += len(chunks)

            # Step 5: Source-to-output mapping
            mapping.append({
                "source": rel(src),
                "output": rel(target),
                "records": len(chunks),
            })

    processing_method = (
        f"{method} text chunks of {size} characters; whitespace collapsed; "
        f"page reference kept per chunk; {total} records"
    )
    return [target], mapping, processing_method


# Step 6: Execute and register the pipeline
run_A = register_and_run(
    "documents",
    SOURCE_A,
    inspect_documents,
    prepare_documents,
    CONFIG_A
)
print("Inspection:", run_A["inspection"])
print("Mapping:", run_A["source_output_mapping"])
print("Cell 8A completed successfully.")

SUCCESS: documents | Inputs: 3 | Outputs: 1
Manifest: data/pipeline_v1/documents/pipeline_manifest.json
Inspection: {'documents': [{'source': 'utm_fyp_psm_guideline.pdf', 'characters': 36087, 'pages': 23}, {'source': 'utm_project_scoping_handbook.pdf', 'characters': 26725, 'pages': 26}, {'source': 'utm_thesis_manual_2015.pdf', 'characters': 93647, 'pages': 80}]}
Mapping: [{'source': 'data/raw/fyp_documents/utm_fyp_psm_guideline.pdf', 'output': 'data/pipeline_v1/documents/prepared_text.jsonl', 'records': 102}, {'source': 'data/raw/fyp_documents/utm_project_scoping_handbook.pdf', 'output': 'data/pipeline_v1/documents/prepared_text.jsonl', 'records': 80}, {'source': 'data/raw/fyp_documents/utm_thesis_manual_2015.pdf', 'output': 'data/pipeline_v1/documents/prepared_text.jsonl', 'records': 238}]
Cell 8A completed successfully.


**Cell 9 — REQUIRED: repeatability and provenance checks.** This is a repeatability smoke test, not full cross-environment reproducibility.

In [15]:
# Cell 9 — REQUIRED: Repeatability and Provenance Verification
# Run after your selected Cell 8A–8F pathway(s).

# Step 1: Ensure at least one pipeline has been registered
assert PIPELINE_RUNS, (
    "No pipeline runs registered. "
    "Run at least one selected Cell 8 pathway first."
)
print("--- Repeatability Verification ---")

# Step 2: Rerun each selected pipeline
for kind, entry in PIPELINE_RUNS.items():
    first = entry["first"]
    # Execute the same pipeline again
    second = entry["runner"]()
    # Step 3: Verify input provenance
    assert first["sources"] == second["sources"], (
        f"Source fingerprint changed for {kind}"
    )
    # Step 4: Verify prepared output fingerprints
    assert first["validation"] == second["validation"], (
        f"Prepared output changed for {kind}"
    )
    # Step 5: Verify source-to-output mapping
    assert (
        first["source_output_mapping"]
        == second["source_output_mapping"]
    ), (
        f"Source-to-output mapping changed for {kind}"
    )
    print(
        f"PASS: {kind} — source fingerprints, "
        "prepared outputs and mappings match"
    )

# Step 6: Display final verification message
print("\nAll selected pipelines passed repeatability smoke checks.")

--- Repeatability Verification ---


SUCCESS: documents | Inputs: 3 | Outputs: 1
Manifest: data/pipeline_v1/documents/pipeline_manifest.json
PASS: documents — source fingerprints, prepared outputs and mappings match

All selected pipelines passed repeatability smoke checks.


**Cell 9B — Part B vs Part C consistency** (addition). The Lab C1 evidence asks you to "explain any differences from Part B". This cell checks that the Part C pipeline reproduces the Part B prepared records exactly. Part C re-implements the splitter, so a match is meaningful evidence.

In [16]:
# Cell 9B — Part B vs Part C consistency check
part_b_path = BASE_DIR / "data" / "processed" / "prepared_text.jsonl"
part_c_path = PIPE_DIR / "documents" / "prepared_text.jsonl"
with part_b_path.open(encoding="utf-8") as fb, part_c_path.open(encoding="utf-8") as fc:
    part_b = [json.loads(line) for line in fb if line.strip()]
    part_c = [json.loads(line) for line in fc if line.strip()]

print("Part B records:", len(part_b), "| Part C records:", len(part_c))
assert part_b == part_c, "Part C output differs from Part B prepared records"
assert sha256_file(part_b_path) == sha256_file(part_c_path)
print("PASS: Part C pipeline reproduces the Part B prepared records exactly "
      "(same records, same SHA-256).")

Part B records: 420 | Part C records: 420
PASS: Part C pipeline reproduces the Part B prepared records exactly (same records, same SHA-256).


### Steps C2–C4 — Documentation and commit

* **C2 — Pipeline diagram:** `docs/week3_pipeline_diagram.png` (editable source: `docs/week3_pipeline_diagram.drawio`). It expands the Week 2 *Departmental FYP Knowledge Sources* component into SOURCE → INGEST → INSPECT → PREPARE/TRANSFORM → VALIDATE OUTPUT → RECORD LINEAGE, with two annotated failure points.
* **C3 — Decisions, stage contracts and traceability:** `docs/week3_pipeline_decisions.md`.
* **C4 — Committed artefacts:**
  * `notebooks/week3_pipeline.ipynb`
  * `docs/week3_data_inventory.md`
  * `data/processed/prepared_text.jsonl`
  * `data/week3_manifest.json`
  * `data/pipeline_v1/documents/`

Embeddings, vector indexing, model calls and data-quality scoring are outside Topic 3 and are left for later topics.